In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 03_global_tech_gold.facts_tables;

In [0]:
from pyspark.sql.functions import col

In [0]:
accounts_df = spark.table("02_global_tech_silver.transformed_tables.transformed_accounts")
pay_df = spark.table("02_global_tech_silver.transformed_tables.transformed_payroll")
gl_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_general_ledgers")
emp_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_employee")
dept_df  = spark.table("02_global_tech_silver.transformed_tables.transformed_departments")
company_df = spark.table("02_global_tech_silver.transformed_tables.transformed_company")


In [0]:
dim_date = spark.table("03_global_tech_gold.dims_tables.dim_date")
dim_company = spark.table("03_global_tech_gold.dims_tables.dim_company")
dim_department = spark.table("03_global_tech_gold.dims_tables.dim_department")
dim_employee = spark.table("03_global_tech_gold.dims_tables.dim_employee")
dim_account = spark.table("03_global_tech_gold.dims_tables.dim_account")

In [0]:
# Financial transactions fact table
fact_general_ledger = (
    gl_df.alias("gl")
    .join(dim_date.alias("d_entry"), col("gl.entry_date") == col("d_entry.date"), "left")
    .join(dim_date.alias("d_posting"), col("gl.posting_date") == col("d_posting.date"), "left")
    .join(dim_company.alias("c"), col("gl.company_id") == col("c.company_id"), "left")
    .join(dim_department.alias("dept"), col("gl.department_id") == col("dept.department_id"), "left")
    .join(dim_account.alias("a"), col("gl.account_id") == col("a.account_id"), "left")
    .select(
        col("gl.gl_sk").alias("gl_key"),
        col("d_entry.date_key").alias("entry_date_key"),
        col("d_posting.date_key").alias("posting_date_key"),
        col("c.company_key"),
        col("dept.department_key"),
        col("a.account_key"),
        
        col("gl.gl_id"),
        col("gl.company_id"),
        col("gl.account_id"),
        col("gl.department_id"),
        
        col("gl.entry_date"),
        col("gl.posting_date"),
        col("gl.fiscal_year"),
        col("gl.fiscal_period"),
        col("d_posting.year_month"),
        
        col("gl.transaction_type"),
        col("gl.reference_number"),
        col("gl.description"),
        
        col("gl.debit_amount"),
        col("gl.credit_amount"),
        
        col("gl.created_by"),
        col("gl.approved_by"),
        col("gl.gl_status"),
        
        col("a.category").alias("account_category")
    )
)

In [0]:
# Payroll Fact Table - 1 row per employee per pay period
fact_payroll = (
    pay_df.alias("p")
    .join(dim_date.alias("d_pay"), col("p.pay_date") == col("d_pay.date"), "left")
    .join(dim_date.alias("d_start"), col("p.pay_period_start") == col("d_start.date"), "left")
    .join(dim_date.alias("d_end"), col("p.pay_period_end") == col("d_end.date"), "left")
    .join(dim_company.alias("c"), col("p.company_id") == col("c.company_id"), "left")
    .join(dim_department.alias("dept"), col("p.department_id") == col("dept.department_id"), "left")
    .join(dim_employee.alias("e"), col("p.employee_id") == col("e.employee_id"), "left")
    .select(
        col("p.payroll_sk").alias("payroll_key"),
        col("d_pay.date_key").alias("pay_date_key"),
        col("d_start.date_key").alias("pay_period_start_date_key"),
        col("d_end.date_key").alias("pay_period_end_date_key"),
        col("c.company_key"),
        col("dept.department_key"),
        col("e.employee_key"),
        
        col("p.payroll_id"),
        col("p.employee_id"),
        col("p.company_id"),
        col("p.department_id"),
        
        col("p.pay_date"),
        col("p.pay_period_start"),
        col("p.pay_period_end"),
        col("d_pay.year_month"),
        
        col("p.bonus"),
        col("p.overtime_pay"),
        col("p.commission"),
        col("p.allowances"),
        col("p.total_compensation"),
        
        col("p.tax_deduction"),
        col("p.social_security"),
        col("p.health_insurance"),
        col("p.retirement_contribution"),
        col("p.other_deductions"),
        col("p.total_deduction"),
        
        col("p.net_salary"),
        col("p.payment_method"),
        col("p.payroll_status")
    )
)

In [0]:
# Save Fact Tables

fact_general_ledger.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_tables.fact_general_ledger")
fact_payroll.write.mode("overwrite").saveAsTable("03_global_tech_gold.facts_tables.fact_payroll")